# NULLXES SHINRA Phase 0 — Colab A100 80GB

Hypothesis check, not full pretrain. RTX 2080 forbidden.
Path: env → param count → BF16 fwd/bwd → stream pilot → tokenizer → pack → 100M tokens.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime → A100"
name = torch.cuda.get_device_name(0)
print(name, "VRAM", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")
assert "2080" not in name, "RTX 2080 is not a SHINRA target"
x = torch.randn(8, 8, device="cuda", dtype=torch.bfloat16)
print("bf16 matmul", (x @ x.T).dtype)

In [ ]:
import os
if not os.path.exists("/content/NULLXES-SHINRA-4B-INSTRUCT"):
    !git clone https://github.com/MagistrTheOne/NULLXES-SHINRA-4B-INSTRUCT.git /content/NULLXES-SHINRA-4B-INSTRUCT
%cd /content/NULLXES-SHINRA-4B-INSTRUCT
!pip install -q -e .
os.environ["PYTHONPATH"] = "/content/NULLXES-SHINRA-4B-INSTRUCT"
os.environ["HF_HOME"] = "/content/shinra_cache"

In [ ]:
!python -m architecture.param_count
!python -m scripts.verify_architecture --seq 256 --batch 1 --attn sdpa --forward --backward

In [ ]:
# Stream PILOT mix. 100M est-tokens for a Colab session, not 5B, not FineWeb 1.3T.
!python -m data.build_pilot \
  --output-dir /content/shinra_scratch/clean/pilot \
  --tokenizer-corpus-dir /content/shinra_scratch/tokcorpus \
  --max-tokens 100000000 \
  --max-disk-gb 150 \
  --cache-dir /content/shinra_cache

In [ ]:
!python -m tokenizer.train_tokenizer \
  --input /content/shinra_scratch/tokcorpus \
  --output-dir /content/NULLXES-SHINRA-4B-INSTRUCT/tokenizer/artifacts \
  --vocab-size 131072 \
  --max-chars 2000000000

In [ ]:
!python -m data.pack \
  --input-dir /content/shinra_scratch/clean/pilot \
  --tokenizer /content/NULLXES-SHINRA-4B-INSTRUCT/tokenizer/artifacts \
  --output-dir /content/shinra_scratch/packed/pilot \
  --sequence-length 2048

In [ ]:
# 100M token hypothesis pretrain on 1× A100 80GB. Not SHINRA-4B-BASE.
!python -m training.pretrain \
  --config configs/shinra_4b.yaml \
  --train-config configs/pretrain_colab_100m.yaml \
  --data-dir /content/shinra_scratch/packed/pilot \
  --tokenizer tokenizer/artifacts \
  --output-dir /content/shinra_scratch/ckpts/hypothesis \
  --attention-implementation sdpa \
  --wandb-project nullxes-shinra \
  --wandb-run-name shinra-colab-100m